In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
driver = webdriver.Chrome()

In [3]:
driver.get("https://merolagani.com/LatestMarket.aspx")

In [10]:
live_trading_div = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "live-trading"))
    )

In [11]:
market_table = live_trading_div.find_element(By.CSS_SELECTOR, "table.live-trading")

In [12]:
table_rows = market_table.find_elements(By.CSS_SELECTOR, "tbody tr")

In [13]:
scraped_data = []

In [15]:
for i in range(len(table_rows)):
    # Re-find the table rows to avoid stale element reference
    current_table_rows = driver.find_elements(By.TAG_NAME, "tr")  # Adjust selector as needed
    
    if i >= len(current_table_rows):
        break
        
    row = current_table_rows[i]
    
    try:
        # Re-find cells within the current row
        cells = row.find_elements(By.TAG_NAME, "td")
        
        if len(cells) < 9:
            continue
            
        scraped_data.append({
            "Symbol": cells[0].text.strip(),
            "LTP": cells[1].text.strip(),
            "% Change": cells[2].text.strip(),
            "Open": cells[3].text.strip(),
            "High": cells[4].text.strip(),
            "Low": cells[5].text.strip(),
            "Qty": cells[6].text.strip(),
            "PClose": cells[7].text.strip(),
            "Diff": cells[8].text.strip(),
        })
    except StaleElementReferenceException:
        # If element becomes stale, skip this iteration
        continue

In [16]:
driver.quit()

In [17]:
df = pd.DataFrame(scraped_data)

In [19]:
print("\n" + "═"*90)
print(f" Successfully scraped {len(df)} live stock rows from Merolagani.")
print("═"*90)
print(df.head(10).to_markdown(index=False)) # Prints the top 10 rows beautifully
print("═"*90)

# Save it to a CSV file if needed
df.to_csv("merolagani_live_market.csv", index=False)


══════════════════════════════════════════════════════════════════════════════════════════
 Successfully scraped 626 live stock rows from Merolagani.
══════════════════════════════════════════════════════════════════════════════════════════
| Symbol   | LTP      |   % Change | Open     | High     | Low      | Qty     | PClose   |   Diff |
|:---------|:---------|-----------:|:---------|:---------|:---------|:--------|:---------|-------:|
| ACLBSL   | 939.00   |      -1.68 | 945.00   | 952.00   | 935.00   | 1,611   | 955.00   |  -16   |
| ADBL     | 305.00   |      -0.33 | 308.00   | 308.00   | 303.00   | 8,138   | 306.00   |   -1   |
| AHL      | 468.60   |      -0.3  | 493.00   | 493.00   | 455.00   | 9,472   | 470.00   |   -1.4 |
| AHPC     | 264.30   |      -0.3  | 265.10   | 267.00   | 264.00   | 49,651  | 265.10   |   -0.8 |
| AKJCL    | 362.60   |      -1.12 | 363.30   | 369.90   | 361.00   | 419,420 | 366.70   |   -4.1 |
| AKPL     | 256.80   |      -0.43 | 269.80   | 269.80   |